In [77]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
pip install shapely

In [4]:
import os
import cv2
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
from google.colab.patches import cv2_imshow

In [67]:
working_directory = '/content/drive/MyDrive/tern_project/'
os.chdir(working_directory)

## Constants

In [6]:
## the convertion from cm to pixel in the drone image
s = 1.8

## the size of all the flag images
image_width = 1280
image_height = 720

## pixel location of the north camera on the drone image
north_cam_x = 11079
north_cam_y = 9341

## pixel location of the south camera on the drone image
south_cam_x = 13199
south_cam_y = 13490

north_cam_loc = np.array([north_cam_x,north_cam_y,0]).reshape(-1,1)
south_cam_loc = np.array([south_cam_x,south_cam_y,0]).reshape(-1,1)

## location in cm - north camera
cam_n = np.array( [0,0,266]).reshape(-1,1)

In [7]:
## finding the location of the south camera in the coordinate of the north camera (in cm)

## the constant to convert from pixel to cm
s_cm = 1/1.8

so1_u = s_cm*north_cam_x
so1_v = s_cm*north_cam_y


# with this matrix I can convert every location in pixel in the drone image to be in the coordinate system of the north camera
##(in cm)
Tr = np.transpose (np.array ([[s_cm,0,0],[0,s_cm,0],[-so1_u,-so1_v,1]]))


south_cam_pix = np.array([south_cam_x, south_cam_y,1])

south_cam_cm = np.dot(Tr , south_cam_pix)

osx = south_cam_cm[0]
osy = south_cam_cm[1]
z = 215

cam_s = np.array([osx,osy,z]).reshape(-1,1)

In [78]:
## The drone image
drone_img = cv2.imread('drone_rotem/Shafiyot_True_Ortho.tif')

In [19]:
## the center of all of the flag images
center = [1280/2, 720/2]

def make_k(f,center):
  K = np.zeros((3,3))
  K[0,0] = f
  K[1,1] = f
  K[2,2] = 1
  K[0,2]= center[0]
  K[1,2] = center[1]
  return K

In [20]:
## calculate the p for the four corners
def calc_p (corners_df,r,k,cam_loc):
  ## cam_loc - the location in cm of the relevant camera

  u = corners_df['U'].values
  v = corners_df['V'].values
  one = np.ones (4)
  u_v = np.column_stack((u,v,one)).T

  p_o = (np.linalg.inv(r))@ np.linalg.inv(k) @u_v

  ## normalize the p-o by dividing the third component (z) to be az the minus z value (height) of the camera
  p_o = p_o/p_o[2]*-cam_loc[2]

  p = p_o + cam_loc
  return (p)


Converting from cm to pixel location on the drone image

In [21]:
def corner_loc_pix (s,cam_loc_pix, corner_real_loc):
  corner_pix_drone = (corner_real_loc*s) + cam_loc_pix.reshape (-1,1)

  corner_pix_drone = corner_pix_drone[0:2,:].T
  corner_pix_drone = corner_pix_drone.astype (np.int32)

  ## change the order of the points to be in a clockwise order
  line_2 =  corner_pix_drone[2,:].copy()
  line_3 = corner_pix_drone [3,:].copy()

  corner_pix_drone[3,:] = line_2
  corner_pix_drone[2,:] = line_3

  return (corner_pix_drone)

### **North Camera**

In [70]:
## reading the relevant values form the text file with all the values for all of the flags

# Specify the file path
#ptz_file_path = 'Chicks/terns-project-chick/RealCoordinatesCalculator/PTZCamValues_191.txt'
ptz_file_path = 'Eyal/RealCoordinatesCalculator/PTZCamValues_191.txt'

# Initialize empty lists to store the data
ptz_nums = []
pitch_values = []
yaw_values = []
zoom_values = []

# Open and read the file line by line
with open(ptz_file_path, 'r') as file:
    for line in file:
        # Split each line by commas and extract values
        parts = line.strip().split(',')
        ptz_num = int(parts[0].split('-')[0].strip('#'))  # Extract ptz_num
        yaw = float(parts[0].split ('-')[1].strip())  # Extract pitch
        pitch = float(parts[1].strip())  # Extract yaw
        zoom = float(parts[2].strip())  # Extract zoom

        # Append the values to their respective lists
        ptz_nums.append(ptz_num)
        pitch_values.append(pitch)
        yaw_values.append(yaw)
        zoom_values.append(zoom)

# Create a DataFrame from the lists
df_ptz = pd.DataFrame({
    'ptz_num': ptz_nums,
    'pitch': pitch_values,
    'yaw': yaw_values,
    'zoom': zoom_values
})

# Display the DataFrame
print(df_ptz)

    ptz_num  pitch     yaw  zoom
0        23  10.68  321.67  44.0
1        24  11.63  313.67  44.0
2        25  12.20  304.80  39.2
3        26  20.36  321.14  13.6
4        27  20.36  298.16  13.6
..      ...    ...     ...   ...
58       81  23.64  352.76  12.0
59       82  46.07  338.31   4.0
60       83  72.23   25.52   4.0
61       84  74.48  260.04   4.0
62       85  78.79  151.99   4.0

[63 rows x 4 columns]


Making adjustment to the camera positions PTZ values

In [71]:
## change the values of the ptz to the values we need for the calibration with the drone image

df_ptz_modi = df_ptz.copy()

df_ptz_modi['pitch'] = df_ptz['pitch'] - 3.1-90
df_ptz_modi['f'] = 186.91 * df_ptz['zoom'] + 514.06


file_path = 'Chicks/terns-project-chick/RealCoordinatesCalculator/PTZCamValues191_mod.txt'

# Save the DataFrame as a text (TXT) file
#df_ptz_modi.to_csv(file_path, sep='\t', index=False)  # Use '\t' as the delimiter for tab-separated values

In [79]:
from shapely.geometry import Polygon

polygons = [];

for index, row in df_ptz_modi.iterrows():
    pitch = row['pitch']
    yaw = row['yaw']
    f = row['f']
    ptz_num = row ['ptz_num']
    #print(f"Row {index + 1}: Pitch={pitch}, Yaw={yaw}, f={f}")

    r = R.from_euler('zyx', [yaw, 0, pitch],degrees=True).as_matrix()
    k = make_k(f, center)


    # Create a DataFrame for the four corners of the flag images
    corners_df = pd.DataFrame({
        'Corner': ['Top Left', 'Top Right', 'Bottom Left', 'Bottom Right'],
        'U': [0, image_width - 1, 0, image_width - 1],
        'V': [0, 0, image_height - 1, image_height - 1]
    })

    # calculate the p (the location in cm) for the four corners
    p = calc_p(corners_df,r,k,cam_n)

    ## calculate the corners inpixel of the drone
    corners = corner_loc_pix(s,north_cam_loc, p)

    cv2.polylines(drone_img, [corners], isClosed=True, color=(0, 0, 255), thickness=7)

    number = str(int(ptz_num))
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 7
    font_color = (0, 0, 255)
    font_thickness = 20
    text_size = cv2.getTextSize(number, font, font_scale, font_thickness)[0]
    text_x = (corners[0, 0] + corners[1, 0] + corners [2,0] + corners [3,0]) // 4
    text_y = (corners[0, 1] + corners[1, 1] + corners [2,1] + corners [3,1]) // 4
    cv2.putText(drone_img, number, (text_x, text_y), font, font_scale, font_color, font_thickness, lineType=cv2.LINE_AA)

    polygons.append({
        'coordinates': Polygon([(corners[0][0], corners[0][1]), (corners[1][0], corners[1][1]),
                        (corners[2][0], corners[2][1]), (corners[3][0], corners[3][1])]),
        'ptz_num': int(number)
    })


scale_factor = 0.1

# Resize the image
resized_image = cv2.resize(drone_img, (0, 0), fx=scale_factor, fy=scale_factor)


cv2_imshow(resized_image)

Output hidden; open in https://colab.research.google.com to view.

### **South Camera**

In [80]:
## reading the relevant values form the text file with all the values for all of the flags

# Specify the file path
#ptz_file_path = 'Eyal/RealCoordinatesCalculator/PTZCamValues_181.txt'
ptz_file_path ='/content/drive/MyDrive/tern_project/Chicks/terns-project-chick/RealCoordinatesCalculator/PTZCamValues_181_2026.txt'

ptz_nums = []
pitch_values = []
yaw_values = []
zoom_values = []

with open(ptz_file_path, 'r') as file:
    for line in file:
        line = line.strip()
        if not line:            # skip blank lines
            continue
        head, rest = line.split(' - ', 1)
        ptz_num = int(head.strip('#'))
        values = rest.split(',')
        yaw   = float(values[0].strip())
        pitch = float(values[1].strip())
        zoom  = float(values[2].strip())

        ptz_nums.append(ptz_num)
        pitch_values.append(pitch)
        yaw_values.append(yaw)
        zoom_values.append(zoom)

df_ptz_s = pd.DataFrame({
    'ptz_num': ptz_nums,
    'pitch': pitch_values,
    'yaw': yaw_values,
    'zoom': zoom_values
})

print(df_ptz_s[:5])

   ptz_num  pitch     yaw   zoom
0       92   6.53  288.56  64.67
1       93   6.22  281.15  72.76
2       94   7.70  274.00  72.76
3       95   7.70  267.12  72.76
4       96   7.70  260.00  72.76


In [81]:

df_ptz_modi_s = df_ptz_s.copy()

df_ptz_modi_s['yaw']   = df_ptz_s['yaw'] + 38.23
df_ptz_modi_s['pitch'] = (df_ptz_s['pitch'] + 1.017) / 1.2299 - 91.47
df_ptz_modi_s['f']     = 127.67 * df_ptz_s['zoom'] + 1120.5

# Flags 116-119 have zoom=0 (negatives set to 0) — use minimum focal length
df_ptz_modi_s.loc[df_ptz_s['zoom'] == 0, 'f'] = 1152.08

print(df_ptz_modi_s[:5])

# Save the DataFrame as a text (TXT) file
df_ptz_modi_s.to_csv(file_path, sep='\t', index=False)  # Use '\t' as the delimiter for tab-separated values

   ptz_num      pitch     yaw   zoom           f
0       92 -85.333729  326.79  64.67   9376.9189
1       93 -85.585782  319.38  72.76  10409.7692
2       94 -84.382432  312.23  72.76  10409.7692
3       95 -84.382432  305.35  72.76  10409.7692
4       96 -84.382432  298.23  72.76  10409.7692


In [82]:

file_path = '/content/drive/MyDrive/tern_project/Chicks/terns-project-chick/RealCoordinatesCalculator/PTZCamValues181_mod.txt'
df_ptz_modi_s[['ptz_num', 'pitch', 'yaw', 'zoom', 'f']].to_csv(file_path, sep='\t', index=False)
print(f'Saved mod file with {len(df_ptz_modi_s)} flags.')

Saved mod file with 46 flags.


In [83]:
#shelved_flags = [120, 121, 137, 138]
shelved_flags = []
for index, row in df_ptz_modi_s.iterrows():
    if row['ptz_num'] in shelved_flags:
        continue

    pitch = row['pitch']
    yaw = row['yaw']
    f = row['f']
    ptz_num = row ['ptz_num']
    #print(f"Row {index + 1}: Pitch={pitch}, Yaw={yaw}, f={f}")

    r = R.from_euler('zyx', [yaw, 0, pitch],degrees=True).as_matrix()
    k = make_k(f, center)

    # Create a DataFrame for the four corners of the flag images
    corners_df = pd.DataFrame({
        'Corner': ['Top Left', 'Top Right', 'Bottom Left', 'Bottom Right'],
        'U': [0, image_width - 1, 0, image_width - 1],
        'V': [0, 0, image_height - 1, image_height - 1]
    })

    # calculate the p (the location in cm) for the four corners
    p = calc_p(corners_df,r,k,cam_n)

    ## calculate the corners inpixel of the drone
    corners = corner_loc_pix(s,south_cam_loc, p)

    cv2.polylines(drone_img, [corners], isClosed=True, color=(255, 0, 0), thickness=7)

    number = str(int(ptz_num))
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 7
    font_color = (255, 0, 0)
    font_thickness = 20
    text_size = cv2.getTextSize(number, font, font_scale, font_thickness)[0]
    text_x = (corners[0, 0] + corners[1, 0] + corners [2,0] + corners [3,0]) // 4
    text_y = (corners[0, 1] + corners[1, 1] + corners [2,1] + corners [3,1]) // 4
    cv2.putText(drone_img, number, (text_x, text_y), font, font_scale, font_color, font_thickness, lineType=cv2.LINE_AA)

scale_factor = 0.1

# Resize the image
resized_image = cv2.resize(drone_img, (0, 0), fx=scale_factor, fy=scale_factor)

cv2_imshow(resized_image)

Output hidden; open in https://colab.research.google.com to view.

In [84]:
save_path = '/content/drive/MyDrive/tern_project/Chicks/terns-project-chick/RealCoordinatesCalculator/flags_on_drone_2026.png'
cv2.imwrite(save_path, resized_image)
print(f'Image saved to {save_path}')

Image saved to /content/drive/MyDrive/tern_project/Chicks/terns-project-chick/RealCoordinatesCalculator/flags_on_drone_2026.png
